# EquityAgent: An Autonomous Agentic Workflow for Investment Research

**GitHub repository:** https://github.com/nicole-rowley-msaai/MS_AAI_520_NLP_GenAI_Final (integration review branch `integration_pawan_ashley`; merge only after approval)

MS in Applied AI, University of San Diego. AAI-520: NLP and Generative AI,
final project.

**Team:** Data lead, NLP lead, Agent lead (Nicole Rowley), Evaluation lead.

EquityAgent researches a stock ticker end to end: it plans the research,
calls market, macro, news and SEC-filing tools, runs a five-step prompt
chain over the news, routes each task to a specialist, writes a brief,
grades and revises it, reflects on the run, and stores lessons that change
the next run's plan. Demo tickers: NVDA, JPM, XOM.

| Part | Contents | Assignment heading |
|---|---|---|
| A | Data check, Kaggle news EDA, classifier evaluation | Agent Functions and Capabilities |
| B | One agent run with every step printed | Agent Design and Workflows |
| C | Baseline comparison, memory ablation, limitations | Evaluation and Iteration |

**Models.** Planner, router and news chain: `gpt-6-luna`. Writer and
optimizer: `claude-sonnet-5`. Evaluator: `gpt-6-sol` (a different family
from the writer, so the draft is not graded by the model that wrote it).
Embeddings: `all-MiniLM-L6-v2`.

**Integration status:** October 18, 2026; 155 local tests passed, and one enhanced NVDA run completed. Live notebook execution remains required.


<!-- AAI520_EVALUATION_INTEGRATION_V1 -->
**Execution instructions:** This is the *single consolidated* Colab notebook for the combined Nicole, Ashley, and Pawan branch. Push `integration_pawan_ashley` to GitHub **before** Colab clones it. Run all cells to generate fresh results. This source notebook contains no current live outputs; historical scores are explicitly identified as earlier observations. The setup uses an isolated Drive repository because the memory experiment deletes that copy's `memory/notes.json`. Do not run it in the team's shared working copy.


## Setup

In [ ]:
# --- Colab setup -----------------------------------------------------
import os
import sys

from google.colab import drive, userdata

os.chdir("/content")
drive.mount("/content/drive")

# To test a pull request, set BRANCH to its branch and REPO to a separate
# Drive folder, so the team's working copy is not touched.
BRANCH = "integration_pawan_ashley"
REPO = "/content/drive/MyDrive/equity-agent-pawan-integration-review"
REPO_URL = "https://github.com/nicole-rowley-msaai/MS_AAI_520_NLP_GenAI_Final.git"

if os.path.isdir(os.path.join(REPO, ".git")):
    !git -C {REPO} fetch -q origin
    !git -C {REPO} checkout -q {BRANCH}
    !git -C {REPO} pull -q
else:
    !git clone -q --branch {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!git log -1 --format='%h %an  %s'

SKIP = r"google-adk\|dependency resolver"
!pip install -q -r requirements.txt 2>&1 | grep -v "{SKIP}"

# Kaggle: newer accounts issue one KAGGLE_API_TOKEN; older ones use
# KAGGLE_USERNAME + KAGGLE_KEY. Either set is enough.
SECRET_KEYS = [
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY",
    "FRED_API_KEY",
    "NEWSAPI_KEY",
    "EDGAR_USER_AGENT",
    "KAGGLE_API_TOKEN",
    "KAGGLE_USERNAME",
    "KAGGLE_KEY",
]
loaded = []
for key in SECRET_KEYS:
    try:
        os.environ[key] = userdata.get(key)
        loaded.append(key)
    except Exception:
        print(f"Missing secret: {key}")
print("Loaded:", loaded)


In [ ]:
import logging
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="huggingface_hub")

from src import display as show  # noqa: E402
from src.config import TICKERS  # noqa: E402
from src.evaluation import (  # noqa: E402
    baseline_brief,
    quick_smoke,
    score_against_findings,
    tool_success_rate,
    unsupported_figures,
)
from src.graph import run_agent  # noqa: E402
from src.memory import store  # noqa: E402
from src.tools import safe_call  # noqa: E402
from src.tools.retrieval import index_filings  # noqa: E402

pd.set_option("display.max_colwidth", 120)


# Part A. Data and NLP

## Agent Functions and Capabilities

The agent's tools and the news prompt chain are built and tested before
the agent uses them. This part checks each data source (yfinance, FRED,
NewsAPI, SEC EDGAR), explores the Kaggle financial-news dataset used to
benchmark the sentiment classifier, and compares the LLM classifier with
a keyword baseline and a supervised TF-IDF baseline, with zero-shot and
few-shot prompts, broken down by annotator agreement. Every tool call
goes through `safe_call`, so a failed API shows as `ok=False` instead of
stopping the notebook.


## A1. Data check: does every tool return data?

Each call goes through `safe_call`, so a failed API shows as `ok=False`
instead of stopping the notebook.

In [ ]:
from src.tools.filings import extract_sections, get_latest_10k
from src.tools.macro import get_macro_snapshot
from src.tools.market import compute_technicals, get_price_history
from src.tools.news import get_news


def status(result):
    """Every ToolResult field except the data itself."""
    return {k: v for k, v in vars(result).items() if k != "data"}


checks = []
spy = safe_call("prices", get_price_history, "SPY")
for ticker in TICKERS:
    prices = safe_call("prices", get_price_history, ticker)
    checks.append({"ticker": ticker, **status(prices)})
    if prices.ok and spy.ok:
        print(ticker, compute_technicals(prices.data, spy.data))

macro = safe_call("fred", get_macro_snapshot)
checks.append({"ticker": "-", **status(macro)})
pd.DataFrame(checks)

In [ ]:
macro.data if macro.ok else macro.error

In [ ]:
news = safe_call("news", get_news, "NVDA", "Nvidia")
print(news.ok, news.meta, news.error)
pd.DataFrame(news.data[:5]) if news.ok else None

**10-K sections.** Three filer quirks surfaced during iteration, and
`src/tools/filings.py` handles each:

- **JPMorgan** embeds its whole annual report in the 10-K file, so "Item 7"
  is only a cross-reference. The extractor falls back to the plain heading
  "Management's discussion and analysis" when the Item 7 span is too short.
- **Exxon** reorganized into ExxonMobil Holdings Corp in mid-2026. The
  ticker now maps to a new SEC registrant with no 10-K, so the loader
  retries the prior entity's CIK from `config.PREDECESSOR_CIKS`.
- **Heavy filers** can push a 10-K off the first page of the SEC
  submissions feed, so the loader follows the paginated files.

The `cik` column below shows which registrant each 10-K came from.

In [ ]:
secs = None
for ticker in TICKERS:
    filing = safe_call("edgar", get_latest_10k, ticker)
    if not filing.ok:
        print(ticker, filing.error)
        continue
    secs = extract_sections(filing.data)
    words = {k: len(v.split()) for k, v in secs.items()}
    print(ticker, "cik", filing.data["cik"], filing.data["filed"], words)

if secs:
    print()
    print(secs["mdna"][:500])

## A2. Kaggle news EDA

The labeled dataset is
[Kaggle: Sentiment Analysis for Financial News](https://www.kaggle.com/datasets/ankurzing/sentiment-analysis-for-financial-news)
(the Financial PhraseBank). The cell below downloads it once with the Kaggle
API; add `KAGGLE_API_TOKEN` (newer Kaggle accounts) or `KAGGLE_USERNAME`
and `KAGGLE_KEY` (older ones) to Colab Secrets, or drop `all-data.csv`
into `data/kaggle/` by hand.

The same sentences ship with the original release in
`data/kaggle/FinancialPhraseBank/` (Malo et al., 2014). Each sentence has
5 to 8 annotations, and the release groups sentences by how strongly the
annotators agreed (100%, over 75%, over 66%, over 50%). The classifier
diagnostics below use those groups to read the classifier's errors.

**Cleaning.** Rows are compared on a normalized key (case-folded,
punctuation removed), which finds near-duplicates that an exact match
misses (Jurafsky & Martin, 2026, Sections 2.4 and 2.6). A sentence that
appears with two different labels has no single gold label, so it is
dropped. The normalized key is used only for matching; the classifier
sees the original text, because case carries sentiment signal (Section
2.6).


In [ ]:
# Download the Kaggle dataset once; it stays in Drive afterward.
from src.config import KAGGLE_CSV

DATASET = "ankurzing/sentiment-analysis-for-financial-news"

if not KAGGLE_CSV.exists():
    # Newer Kaggle accounts issue one API token; older ones a username and
    # key. Load whichever is in Colab Secrets.
    for name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            os.environ.setdefault(name, userdata.get(name))
        except Exception:
            pass
    !pip install -q -U kaggle
    !kaggle datasets download -d {DATASET} -p data/kaggle --unzip
print("Kaggle CSV present:", KAGGLE_CSV.exists())


In [ ]:
from src.chains.kaggle import (
    LABELS, clean_kaggle, cleaning_report, load_kaggle, train_test,
)

raw = load_kaggle()
print("Rows:", len(raw), "| Columns:", list(raw.columns))
print("Missing values:", raw.isna().sum().to_dict())
print("Cleaning:", cleaning_report(raw))
df = clean_kaggle(raw)
print("Rows after cleaning:", len(df))


In [ ]:
counts = df["sentiment"].value_counts().reindex(LABELS)
print((counts / counts.sum()).round(3))

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(counts.index, counts.values, color="#3b6fb6", width=0.6)
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
ax.set_title("Label balance: mostly neutral")
ax.set_ylabel("Headlines")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
df["words"] = df["headline"].str.split().str.len()
print(df.groupby("sentiment")["words"].describe().round(1))

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(df["words"], bins=30, color="#3b6fb6")
ax.set_title("Headline length (words)")
ax.set_xlabel("Words")
ax.set_ylabel("Headlines")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# Fixed seed, stratified. Few-shot examples and the supervised baseline
# use only `train`; `test` is never shown to a model before scoring.
train, test = train_test(df, n=300)
test.to_csv("data/kaggle/test_sample.csv", index=False)
print("Train:", len(train), "| Test:", len(test))
test["sentiment"].value_counts()


## A3. Classifier evaluation: keyword baseline vs LLM

The LLM classifier is scored on the fixed 300-headline held-out sample with
a sentiment-only prompt. The live news chain uses a longer prompt that also
asks for relevance, but the Kaggle headlines have no target company, so that
instruction would only add noise here. Because LLM output varies between
runs (about 2 points of accuracy at n = 300), the score is the mean of
three runs.

In [ ]:
from src.chains.kaggle import keyword_baseline, score, to_articles
from src.chains.news_chain import (
    CLASSIFY_SENTIMENT_SYSTEM,
    classify,
    coverage,
)

METRICS = ("accuracy", "macro_f1")
N_RUNS = 3  # LLM output varies run to run; report the mean, not one draw

keyword_pred = list(test["headline"].map(keyword_baseline))
base = score(test["sentiment"], keyword_pred)
print("Keyword baseline:", {k: base[k] for k in METRICS})

articles = to_articles(test)
runs, zero_preds = [], []
for r in range(N_RUNS):
    labels = classify(articles, system=CLASSIFY_SENTIMENT_SYSTEM)
    missing = coverage(labels, articles)["missing_ids"]
    pred = [
        labels.get(i, {}).get("sentiment", "neutral")
        for i in range(len(test))
    ]
    zero_preds.append(pred)
    runs.append(score(test["sentiment"], pred))
    print(
        f"LLM run {r + 1}:",
        {k: runs[-1][k] for k in METRICS},
        f"| unlabeled: {len(missing)}",
    )

mean = {k: round(sum(s[k] for s in runs) / N_RUNS, 3) for k in METRICS}
print("LLM mean of", N_RUNS, "runs:", mean)

llm = runs[-1]  # last run's per-class report and confusion matrix
print(llm["report"])


In [ ]:
cm = llm["confusion"]

fig, ax = plt.subplots(figsize=(4, 3.5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), LABELS)
ax.set_yticks(range(3), LABELS)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
for i in range(3):
    for j in range(3):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, cm[i, j], ha="center", va="center", color=color)
ax.set_title("LLM classifier confusion matrix")
plt.show()

## A4. Classifier diagnostics: supervised baseline, few-shot prompts, agreement

Three checks on the zero-shot LLM classifier above:

1. **A supervised baseline.** A pretrained model can be applied to a task
   by prompting it or by training on labeled data (Jurafsky & Martin, 2026,
   Section 9.4). TF-IDF with logistic regression, trained on the same
   train split, stands in for the second, so the LLM is compared with a
   model that learned from these labels and not only with keywords.
2. **Few-shot prompting.** Two demonstrations per label are added to the
   prompt, drawn from the train split, as Brown et al. (2020, Section 2.4)
   draw conditioning examples from the training set and never from the
   test set. Demonstrations come from sentences every annotator agreed
   on, and K is fixed in advance rather than tuned on the test set. The
   sentiment definition in the prompt now restates the PhraseBank
   annotation guideline: judge only what the sentence states, from an
   investor's view of the stock price (Malo et al., 2014).
3. **Majority vote across runs.** The three runs per prompt are combined
   by majority vote, the self-consistency idea of keeping the answer most
   samples agree on (Wang et al., 2023). It reuses the runs already made,
   so it adds no API calls.

Finally, accuracy is broken down by annotator agreement. If the model is
far more accurate where every annotator agreed, many remaining "errors"
are sentences that people also labeled differently.


In [ ]:
from src.chains.kaggle import (
    fewshot_examples,
    load_agreement,
    majority_vote,
    score_by_agreement,
    tfidf_baseline,
)

agreement = load_agreement()
print("Sentences with an agreement band:", len(agreement))

tfidf_pred = tfidf_baseline(train, test)

examples = fewshot_examples(train, per_label=2, agreement=agreement)
print("Few-shot demonstrations (train split, unanimous labels):")
for text, label in examples:
    print(f"  {label:>8}: {text[:90]}")

few_preds = []
for r in range(N_RUNS):
    labels = classify(
        articles, system=CLASSIFY_SENTIMENT_SYSTEM, examples=examples
    )
    few_preds.append(
        [
            labels.get(i, {}).get("sentiment", "neutral")
            for i in range(len(test))
        ]
    )

candidates = {
    "keyword": [keyword_pred],
    "tf-idf + logistic regression": [tfidf_pred],
    "LLM zero-shot": zero_preds,
    "LLM few-shot": few_preds,
}
rows = []
for name, preds in candidates.items():
    per_run = [score(test["sentiment"], p) for p in preds]
    vote = score(test["sentiment"], majority_vote(preds))
    rows.append(
        {
            "classifier": name,
            "runs": len(preds),
            **{
                f"{m} (mean)": round(
                    sum(s[m] for s in per_run) / len(preds), 3
                )
                for m in METRICS
            },
            **{f"{m} (vote)": vote[m] for m in METRICS},
        }
    )
comparison = pd.DataFrame(rows)
comparison


In [ ]:
final_preds = {
    "tf-idf + LR": tfidf_pred,
    "LLM zero-shot (vote)": majority_vote(zero_preds),
    "LLM few-shot (vote)": majority_vote(few_preds),
}
by_band = pd.concat(
    score_by_agreement(test, p, agreement).assign(classifier=name)
    for name, p in final_preds.items()
)
band_order = ["100%", "75-99%", "66-74%", "50-65%"]
acc = by_band.pivot(index="band", columns="classifier", values="accuracy")
acc = acc.reindex([b for b in band_order if b in acc.index])
counts = by_band.drop_duplicates("band").set_index("band")["n"]
print("Test sentences per band:", counts.reindex(acc.index).to_dict())
display(acc)

fig, ax = plt.subplots(figsize=(6, 3.2))
width = 0.8 / len(acc.columns)
for j, name in enumerate(acc.columns):
    offset = (j - (len(acc.columns) - 1) / 2) * width
    xs = [i + offset for i in range(len(acc))]
    ax.bar(xs, acc[name], width, label=name)
ax.set_xticks(range(len(acc)), acc.index)
ax.set_xlabel("Annotator agreement")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Accuracy falls where annotators disagreed")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8)
plt.show()


**Reading the results (run of October 18, 2026).** The zero-shot LLM
reached 0.834 accuracy and 0.838 macro-F1 (mean of three runs), against
0.793 and 0.771 for TF-IDF with logistic regression and 0.687 and 0.575 for
the keyword baseline. Neutral recall rose from 0.72 in the October 14 run
to 0.85. That run used the earlier prompt and a test sample drawn before
the extra de-duplication, so the gain combines the guideline-based
sentiment definition with the cleaning. Few-shot demonstrations did not
help (0.827 and 0.833): once the label definition is explicit, two
examples per class add little. The majority vote raised the zero-shot
scores slightly, to 0.840 and 0.843.

Every classifier loses accuracy as annotator agreement falls. The
zero-shot LLM is right on 95% of the sentences every annotator agreed on
but on 47% of those where only 50-65% agreed, and TF-IDF and few-shot fall
to similar levels there. The lowest band is a ceiling set by the labels,
not only by the model.

One caveat: the Financial PhraseBank has been public since 2014, so the
LLM may have seen these sentences during pre-training. Brown et al.
(2020, Section 4) show that such test-set contamination can inflate
benchmark scores, so the LLM's numbers may be optimistic for live news.


# Part B. Agent run

## Agent Design and Workflows

The agent is a LangGraph state graph (`src/graph.py`):

```
ticker -> plan -> route -> specialists -> synthesize -> evaluate --pass--> reflect -> memory
                                                            |                 ^
                                                            +--refine--> optimize
```

| Requirement | Where | Printed below |
|---|---|---|
| Plans research steps | `agents/planner.py` | B2 |
| Uses tools dynamically | `agents/toolkit.py`, `tools/` | B3 |
| Routing | `agents/router.py` | B4 |
| Prompt chaining | `chains/news_chain.py` | B5 |
| Evaluator-optimizer | `agents/evaluator.py` | B7 |
| Self-reflection | `agents/evaluator.py: reflect` | B9 |
| Learns across runs | `memory/store.py` | B10 |

This part runs the agent once on NVDA and prints the output of every step
from the run trace, then runs it a second time to show memory in use.
Memory is cleared first so B2 is a true first run.


## B1. Setup check: every tool returns data


In [ ]:
print("Chunks indexed:", index_filings(TICKERS))
pd.DataFrame({t: quick_smoke(t) for t in TICKERS})

## B2. Research plan

The planner reads memory first. On the first run of a ticker there are no
notes; later runs show the lessons it used.

In [ ]:
# Start from empty memory so the first run below is a true first run.
store.NOTES_PATH.unlink(missing_ok=True)
print("Lessons in memory:", len(store.load_notes()))


In [ ]:
TICKER = "NVDA"
notes_before = store.load_notes()

report, trace, state = run_agent(TICKER, reviewed_evaluation=True)

print("Sector:", state["sector"])
print("Hypothesis:", state["plan"].hypothesis)
show.plan_table(trace)

In [ ]:
show.memory_read(trace)

## B3. Tool calls (dynamic tool use)

The plan, not the code, decides which tools run and with what query.
The three specialists run in parallel, so the log is grouped by the
specialist (`route`) that made each call.


In [ ]:
show.tool_log(trace)

## B4. Routing decisions

In [ ]:
show.routing_table(trace)

## B5. News prompt chain: output of each step

The digest's `net_sentiment` is not the model's opinion: the chain counts
the per-article labels from the classify step and applies majority rule
with a 15-point margin (`sentiment_counts` is printed with the summary).
Empty `figures` lists in the extract step are expected for headline-only
items; the extractor only reports numbers that appear in the text.


In [ ]:
show.news_chain_steps(trace)

## B6. Specialist findings

In [ ]:
show.findings_table(trace)

## B7. Evaluation and optimization: original rubric plus reviewed quality gates

The existing evaluator (`src/agents/evaluator.py`) scores five dimensions on a
1–5 scale and provides specific critiques. In this integrated run, the
**same LLM evaluator** supplies the scores, but the Evaluation Lead extension
adds independent acceptance checks for **accuracy**, **evidence**, and
explicitly recognized numeric contradictions. It also retains the strongest
accepted candidate rather than assuming the latest revision is best.

`reviewed_evaluation=True` activates the extension in `src/graph.py`.
Unknown or unparseable financial claims do **not** automatically fail the
report; numeric contradictions in recognized claims do. The parser matches
only a narrow set of metric/period/unit patterns, so a clean audit is **not**
proof that the entire brief is numerically correct.

The graph preserves `state["history"]` for the original score table and plot.
An initial candidate may fail the policy even when its average score reaches
4.0. The accepted result and selected revision are recorded separately.


In [ ]:
show.evaluation_history(state)

In [ ]:
show.critiques(state)

In [ ]:
show.plot_scores(state)
plt.show()

In [ ]:
# Live quality-gate audit for the Part B research run.
# Uses the evaluated state already generated above; no additional API calls.
review_rows = []
for item in state["history"]:
    scores = item["scores"]
    review_rows.append({
        "iteration": item["iteration"],
        "average": item["average"],
        "accuracy": scores["accuracy"],
        "evidence": scores["evidence"],
        "candidate_pass": item.get("accepted"),
        "failures": ", ".join(item.get("failures", [])) or "none",
        "numeric_checked": item.get("numeric_checked", 0),
        "numeric_contradicted": item.get("numeric_contradicted", 0),
        "numeric_unverified": item.get("numeric_unverified", 0),
        "selected": item.get("selected", False),
    })
display(pd.DataFrame(review_rows))
print("Selected iteration:", state["selected_iteration"])
print("Accepted:", state["accepted"], "| Stop reason:", state["stop_reason"])
print("Selected rubric score:", state["evaluation"].average)
print("Numeric matches in selected brief:", len(state["numeric_audit"].checks))
print("Warning: numbers outside the limited parser remain unchecked.")
display(pd.DataFrame(state["numeric_audit"].rows()))


## B8. Final report

In [ ]:
display(show.report_markdown(report))

## B9. Reflection (self-assessment of output and process)

In [ ]:
show.reflection_view(state)

## B10. Memory: what this run taught the agent

New lessons written to `memory/notes.json`, then a second run on the same
ticker to show the planner reading them.

In [ ]:
show.memory_diff(trace)

In [ ]:
report2, trace2, state2 = run_agent(TICKER, reviewed_evaluation=True)
print("Lessons the planner read on run 2:")
display(show.memory_read(trace2))
print("Run 1 steps:", [s.tool for s in state["plan"].steps])
print("Run 2 steps:", [s.tool for s in state2["plan"].steps])
print("Run 1 final score:", state["evaluation"].average,
      "| Run 2 final score:", state2["evaluation"].average)

# Part C. Evaluation

## Evaluation and Iteration

Part B showed one evaluator-optimizer loop. This part measures the agent
against alternatives, all scored with the same rubric and the same
findings so the comparison is fair.

The numeric spot-check (`unsupported_*`) counts figures in a brief that
appear in no finding. A figure the writer labels as calculated (for
example "a calculated ~65% increase") is exempt: the evaluator asks for
that label, and such figures are arithmetic on the findings rather than
claims needing their own source. The baseline never sees the findings,
so its evidence and accuracy scores are structurally low; the comparison
measures whether tools and the loop produce a grounded brief.

In this branch, Part B and the C1 baseline experiment use the reviewed policy on the same LLM rubric. Policy C2 is a replay on those recorded observations, not an independent second model run.


## C1. Three comparisons

Three comparisons, all scored with the same rubric and the same findings:

1. **Baseline vs agent:** a single prompt with no tools, no plan, no loop.
2. **Numeric spot-check:** figures in each brief that appear in no finding.
3. **Memory on vs off:** the same ticker with memory disabled.

In [ ]:
rows = []
for t in TICKERS:
    rep, tr, st = run_agent(t, reviewed_evaluation=True)
    base = baseline_brief(t)
    base_score = score_against_findings(base, st["findings"])
    findings = st["findings"]
    rows.append(
        {
            "ticker": t,
            "baseline_avg": base_score["average"],
            "agent_first_draft": st["history"][0]["average"],
            "agent_final": st["evaluation"].average,
            "accepted": st["accepted"],
            "selected_iteration": st["selected_iteration"],
            "numeric_checked": len(st["numeric_audit"].checks),
            "numeric_contradicted": len(st["numeric_audit"].contradictions),
            "numeric_unverified": len(st["numeric_audit"].unverified),
            "iterations": len(st["history"]),
            "tool_success": tool_success_rate(tr),
            "unsupported_baseline": len(unsupported_figures(base, findings)),
            "unsupported_agent": len(
                unsupported_figures(st["draft"], findings)
            ),
        }
    )
results = pd.DataFrame(rows)
results

In [ ]:
from pathlib import Path
table1_path = Path("Results/table1_integrated_live_run.csv")
table1_path.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(table1_path, index=False)
print("Table 1 saved:", table1_path.resolve())
print("These are fresh integrated-run measurements; copy these values to the report.")
print(results.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
x = list(range(len(results)))
w = 0.27
series = [
    ("baseline_avg", "baseline", "#b5b5b5", -w),
    ("agent_first_draft", "agent, first draft", "#8fb3e0", 0),
    ("agent_final", "agent, final", "#3b6fb6", w),
]
for col, label, color, offset in series:
    ax.bar([i + offset for i in x], results[col], w, label=label, color=color)
ax.set_xticks(x, results["ticker"])
ax.set_ylim(1, 5)
ax.set_ylabel("Rubric average (1-5)")
ax.set_title("Rubric comparison: no-tools baseline and integrated agent")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
_, tr_off, st_off = run_agent(TICKER, use_memory=False, reviewed_evaluation=True)
_, tr_on, st_on = run_agent(TICKER, use_memory=True, reviewed_evaluation=True)
pd.DataFrame([
    {"memory": "off", "plan_steps": len(st_off["plan"].steps),
     "notes_read": len(tr_off.select("memory", "read")[0].output),
     "final_score": st_off["evaluation"].average},
    {"memory": "on", "plan_steps": len(st_on["plan"].steps),
     "notes_read": len(tr_on.select("memory", "read")[0].output),
     "final_score": st_on["evaluation"].average},
])

In [ ]:
show.timing_table(trace)

## C2. Controlled comparison of evaluator acceptance policies

**Experiment design.** Both policies use the *same* specialist findings,
report revisions, and rubric scores from the enhanced Part B run. Only the
**decision rule** changes. The original evaluator stops after the required
minimum revision when the rubric average is at least its pass threshold
(or the evaluation budget is exhausted). The reviewed evaluator adds
accuracy and evidence floors and deterministic checks on recognized
financial claims, and can select an earlier accepted revision if later
revisions regress.

This is a **paired policy replay**, **not** a separate original-agent run and
not a controlled comparison of two independent LLM judges. If the original
policy would stop earlier, later rows are marked as not reached in that
counterfactual path. The observed trajectory was generated by the enhanced
optimizer and may not be the trajectory the original optimizer would take.


In [ ]:
# Replay Nicole's original stopping rule on the SAME recorded rubric scores.
# This never calls an LLM and does not manufacture a second model output.
from src.agents import evaluator as original_eval

original_stopped = False
replay_rows = []
for index, item in enumerate(state["history"]):
    iteration = item["iteration"]
    avg = item["average"]
    if original_stopped:
        original_action = "not reached"
    else:
        original_pass = (iteration >= original_eval.MIN_ITERATIONS
                         and avg >= original_eval.PASS_THRESHOLD)
        original_budget = iteration + 1 >= original_eval.MAX_ITERATIONS
        original_action = ("accept" if original_pass else
                           "exhausted" if original_budget else "revise")
        original_stopped = original_action != "revise"

    # Action the reviewed graph ACTUALLY took at the time of this row.
    last = index == len(state["history"]) - 1
    reviewed_action = ("revise" if not last else
                       "accept" if state["accepted"] else "exhausted")
    replay_rows.append({
        "iteration": iteration,
        "rubric_average": avg,
        "accuracy": item["scores"]["accuracy"],
        "evidence": item["scores"]["evidence"],
        "original_policy": original_action,
        "reviewed_policy": reviewed_action,
        "reviewed_failures": ", ".join(item.get("failures", [])) or "none",
    })
replay = pd.DataFrame(replay_rows)
display(replay)
print("Reviewed selected iteration:", state["selected_iteration"])
print("Original policy and reviewed policy use the same rubric scores.")


### Policy difference beyond this one observed trajectory

The two stopping rules can make identical decisions on a good report; that
outcome does not mean the additional controls are redundant. The next
illustration is **hypothetical** and explicitly is not an observed agent run.


In [ ]:
# Deliberately hypothetical case to show the acceptance-policy distinction.
# The rubric scores are invented for this illustration, NOT measured results.
illustration = pd.DataFrame([{
    "condition": "After minimum revision; average 4.4, accuracy 2, evidence 5",
    "original_policy": "accept (average >= 4.0)",
    "reviewed_policy": "reject (accuracy below policy floor)",
    "evidence_type": "hypothetical, not empirical",
}])
display(illustration)


### Previously observed integration checkpoint (October 18, 2026)

The team previously ran the enhanced evaluator on **NVDA** from the command
line with Ashley's NLP branch integrated. The terminal showed:

| Iteration | Average | Accuracy | Evidence | Recognized checks | Unverified checks | Contradictions | Review outcome |
|---|---:|---:|---:|---:|---:|---:|---|
| 0 | 4.0 | 3 | 4 | 4 | 2 | 0 | Reject: accuracy below threshold |
| 1 | 4.8 | 5 | 5 | 2 | 0 | 0 | Accept; selected iteration 1 |

Replaying the **original** policy on those *recorded scores* also led to
iteration 1. No draft and specialist-finding snapshot was saved from that
command-line run, so the old and new evaluator cannot be independently
regraded on the exact former report. The **live tables above**, not these
historical numbers, should be used for the final submission's results.

The 155 passing local tests and the successful NVDA run are separate
checkpoints. Unit tests and a single live run do not establish that all
financial figures are accurate or that all tickers will achieve the same scores.


## C3. Limitations

- Figures come from free data sources (`yfinance` is unofficial; NewsAPI
  caps at 100 articles) and are cached, so a run reflects the cache date.
- The evaluator is an LLM and its scores vary by about 0.2 between runs.
- The agent has no view of future events, and the brief is analysis only,
  not investment advice.
- The numeric audit recognizes only a small fraction of arbitrary prose and cannot certify the brief.
- A gate replay compares decisions on recorded scores, not two independent LLM evaluations.
- The LLM reviewer is nondeterministic and can produce different results in another run.


## Summary and reproducibility

- **End-to-end agent:** Original LangGraph research, routing, tools,
  synthesis, rubric evaluation, reflection, and memory are retained.
- **NLP:** Ashley's classifier updates are inherited from the integration
  branch; classifier benchmark values must be regenerated if inputs change.
- **Evaluation Lead:** The enhanced graph now audits a narrow set of financial
  claims, applies separate accuracy and evidence floors, provides feedback
  to revision, and selects the best accepted draft. It reports an exhausted
  budget as *not accepted*, not as success.
- **Policy comparison:** Original and reviewed rules are replayed against the
  same scored drafts. This isolates the **gate decision** only, not a causal
  improvement in LLM-generated brief quality.
- **Baselines:** The no-tools baseline in C1 measures the effect of accessing
  the research workflow, not solely the revision loop. The comparison in C2
  is a policy ablation; these answer different research questions.
- **Numeric limits:** Claims outside the restrictive parser are not checked;
  rounding, periods, units, conflicting filings, and source reliability
  require additional validation or manual review.
- **Run artifacts:** Execute **every code cell in order** and save the
  resulting `.ipynb` with cell outputs plus the HTML export. Update the
  supplemental report tables with those executed results, not with the
  previously recorded October 18 checkpoint.
- **Development provenance:** Evaluation extension and associated synthetic
  tests were developed with substantial AI assistance and were reviewed
  and tested by the Evaluation Lead. Cite sources and disclose AI assistance
  according to the university's academic integrity requirements.
